In [1]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline
import torch

Load a smaller, BERT-variant named **DistilBERT** from Hugging Face that is under the name `"distilbert-base-uncased"`.

Here's a table summarizing the difference between BERT and DistilBERT:

| Model               | Layers | Hidden Size | Parameters (approx.) | Relative Size | Summary                                                                                                                              |
| ------------------- | ------ | ----------- | -------------------- | ------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **BERT-base**       | 12     | 768         | 110 million     | 100%          | Original full BERT model trained with next sentence prediction and masked language modeling.                                       |
| **DistilBERT-base** | 6      | 768         | 66 million      | ~60%          | Distilled version of BERT-base with half the layers and no next sentence prediction objective. Retains ~97% of BERT’s performance. |

In [ ]:
text = "The playful fox chases butterflies"

distilbert_model_name = "distilbert-base-uncased"
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_model_name)
distilbert_model = AutoModel.from_pretrained(distilbert_model_name)

distilbert_inputs = distilbert_tokenizer(text, padding=True, truncation=True, return_tensors="pt")

# Show output - distilbert inputs
print(distilbert_inputs)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[  101,  1996, 18378,  4419, 29515, 15023,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
with torch.no_grad():
    distilbert_outputs = distilbert_model(**distilbert_inputs)
    
distilbert_last_hidden_state = distilbert_outputs.last_hidden_state

# Show output - distilbert rich, context-aware token vectors
print("Last Hidden State Shape:", distilbert_last_hidden_state.shape)
print(distilbert_last_hidden_state)

Last Hidden State Shape: torch.Size([1, 7, 768])
tensor([[[-0.3895, -0.0209, -0.0671,  ..., -0.0796,  0.3997,  0.2861],
         [-0.5013,  0.0928, -0.4862,  ..., -0.0674,  0.8544, -0.1640],
         [-0.1955,  0.3190,  0.1197,  ..., -0.1122,  0.2617, -0.0246],
         ...,
         [-0.0636,  0.1912,  0.3354,  ..., -0.2363,  0.1154,  0.2358],
         [-0.3176, -0.0186, -0.2126,  ..., -0.1341,  0.1623,  0.0115],
         [ 0.9369,  0.3148, -0.3745,  ...,  0.0718, -0.4303, -0.1223]]])


Load a smaller, GPT-variant named **DistilGPT-2** from Hugging Face under the name `"distilgpt2"`. 

Here's a table summarizing the difference between GPT-2 and DistilGPT-2:

| Model             | Layers | Hidden Size | Parameters (approx.) | Relative Size | Summary                                                                                                                          |
| ----------------- | ------ | ----------- | -------------------- | ------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| **GPT-2 (small)** | 12     | 768         | 124 million     | 100%          | Original GPT-2 model trained for autoregressive (left-to-right) language modeling. Generates text by predicting the next token.          |
| **DistilGPT-2**   | 6      | 768         | 82 million      | ~66%          | Distilled version of GPT-2 with half the layers. Retains most of GPT-2’s text-generation quality while being faster and smaller. |

In [ ]:
prompt = "The playful fox chases butterflies"

distilgpt_model_name = "distilgpt2"
distilgpt_tokenizer = AutoTokenizer.from_pretrained(distilgpt_model_name)
distilgpt_model = AutoModelForCausalLM.from_pretrained(distilgpt_model_name)

distilgpt_inputs = distilgpt_tokenizer(prompt, return_tensors="pt")

# Show output - GPT2 inputs
print(distilgpt_inputs)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

{'input_ids': tensor([[  464, 34264, 21831,   442,  1386, 44571]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [ ]:
torch.manual_seed(0)

with torch.no_grad():
    distilgpt_outputs = distilgpt_model.generate(**distilgpt_inputs, max_new_tokens=50, do_sample=True) # type: ignore[attr-defined]

generated_text = distilgpt_tokenizer.decode(distilgpt_outputs[0])

# Show output - generated text
print(generated_text)

The playful fox chases butterflies along the way. He‪ is the best fox in the room, but unfortunately the animal is too weak to be frightened. He‪ can‪ be in a position he can‪ be left alone when the fox is still standing.


In [ ]:
# Using Microsoft DialoGPT-small to answer a question about the closest star to Earth
# The expected answer should be "The Sun"
# This is an additional example not originally included in the main tutorial from Codecademy

ms_tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
ms_model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small", device_map="auto")
messages = [
    {"role": "user", "content": "Which is the closest star to Earth?"},
]
inputs = ms_tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(ms_model.device)

outputs = ms_model.generate(**inputs, max_new_tokens=256) # type: ignore[attr-defined]
print(ms_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I think the closest star to earth is the sun .<|endoftext|>
